# Phase 2: Legal NER - Legal Transformer with LoRMA

## Task
Legal named entity recognition (NER): find entities (names, courts, statutes, dates and so on) in legal judgments. Every word gets a B/I/O tag, and predictions are scored with strict macro F1 (start, end and label must all match).

Model: the pre-trained Legal Transformer fine-tuned with LoRMA (a small multiplicative low-rank update of its weights), with a BiGRU and a linear tag layer on top.

## Input
- Phase-2 NER data; downloaded from Hugging Face if not attached.
- Legal Transformer checkpoint .pt and its tokenizer .json.
- Optional: the output of an earlier session to continue from (RESUME_FROM).

## Flow of the notebook (from input to output)
1. Set the settings and file paths.
2. Find the checkpoint and tokenizer files.
3. Write the model code, the NER pipeline, the fine-tuning methods, the decoding options and the long-run training engine as .py files.
4. Load the NER documents and make a document-level train / hold-out split.
5. Build the training settings and learning-rate groups.
6. Load the pre-trained model and tokenize the documents.
7. Set up the LoRMA model for each setting (rank, learning rate, with or without the document header added in front of every window).
8. Train every setting on the train part and score it on the 14 hold-out documents every epoch; keep the best, final and averaged weights.
9. Retrain the two best settings on all 70 documents with several seeds and average their predictions.
10. Decode the test predictions with each decoding option (Viterbi, no-entity penalty, top-k, top-p) and write one zip per option.
11. Show a results table and plots, then zip all outputs.

## Output
- Submission zips; each file name says the model, the weight set (best, final or avg) and the decoding option.
- Saved hold-out and test probabilities (.npz) for ensembling.
- One zip with all outputs.

Every run saves a checkpoint every 30 minutes. If the session ends, set RESUME_FROM and run again: finished runs are skipped and an interrupted run continues.

In [1]:
# Settings and file paths
AUTO_DOWNLOAD = True
SMOKE = False
SMOKE_CHARS = 12000
SEED = 42
OUT = "/kaggle/working"
RESUME_FROM = None
SAVE_EVERY_MIN = 30
N_VAL_DOCS = 14
WIN_TOKENS = 512
STRIDE = 256
TTA_STRIDES = [256, 192, 320, 128]
DOUBLE_PASS = False
BIDIRECTIONAL = False
GRU_HIDDEN = 256
HEAD_DROPOUT = 0.3
LABEL_SMOOTH = 0.05
RARE_DUP = 2
CLASS_POWER = 0.5
EPOCHS = 20
SWA_LAST = 4
PSEUDO = False
PSEUDO_THR = 0.9
PSEUDO_SEEDS = 2
PATIENCE = 0
MICRO_BS = 8
ACCUM = 2
EVAL_FROM = 2
LR_HEAD = 2e-3
WEIGHT_DECAY = 0.01

NAME = "lorma_lt_best"
CTX_TOKENS = 128
FULL_SEEDS = [0, 1]
EPOCHS = 20
PSEUDO = False
PSEUDO_THR = 0.9
PSEUDO_SEEDS = 2
N_FINAL_VARIANTS = 1
VARIANTS = [dict(tag="r32_ctx", rank=32, lr=5e-4, ctx=CTX_TOKENS), dict(tag="r32_ctx0", rank=32, lr=5e-4, ctx=0)]
CKPT = "/kaggle/input/datasets/shiva091ty/legal-transformer-final-pt-p2/legal_transformer_random_moredata_best_random_90_10.pt"
TOK = "/kaggle/input/datasets/shiva091ty/transformer-tokenizer-p2/tokenizer_legal_transformer_random_moredata_random_90_10.json"

In [2]:
# Find the model checkpoint and tokenizer files
import os

# It takes exts (file extensions) as input, lists files in the input folder with the given extensions and returns a sorted list of file paths.
def _walk(exts):
    out = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for n in dirs + files:
            p = os.path.join(root, n)
            if p.lower().endswith(exts): out.append(p)
    return sorted(set(out))

# It takes label (name to print), explicit (path set by hand), exts (file extensions), keywords (words the file name should contain), require (words that must be in the name), exclude (words to skip) and defaults (default paths) as input, finds the right input file to use and returns a path of the chosen file.
def pick(label, explicit, exts, keywords=(), require=(), exclude=(), defaults=()):
    if explicit: return explicit
    for d in defaults:
        if os.path.exists(d):
            print(f"{label}: {d}"); return d
    c = [p for p in _walk(exts) if all(r in p.lower() for r in require) and not any(x in p.lower() for x in exclude)]
    k = [p for p in c if any(w in p.lower() for w in keywords)] or c
    if len(k) == 1:
        print(f"{label}: {k[0]}"); return k[0]
    seen = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.count(os.sep) <= 5: seen += [os.path.join(root, n) for n in files + dirs][:12]
    raise RuntimeError(f"{label}: found {len(k)} candidate files {k[:8]}.\n"
                       f"Attach the file with 'Add Input' (or set the path in the Settings cell).\n"
                       f"What is currently attached under /kaggle/input: {seen[:40] if seen else 'NOTHING'}")

CKPT = pick("checkpoint", CKPT, (".pt",), keywords=("transformer",), exclude=("last",), defaults=(CKPT,))
TOK = pick("tokenizer", TOK, (".json",), keywords=("token",), require=("token",), exclude=("fold",), defaults=(TOK,))

In [3]:
%%writefile arch_legal_transformer.py
# Legal Transformer model code (saved as arch_legal_transformer.py)
import json
import math
import os
import re as _re
import shutil
import tempfile
import time as _time
import zipfile
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "legal_transformer"
TRAINED_SEQ_TOKENS = 640
NOTEBOOK_ROPE_LEN = TRAINED_SEQ_TOKENS + 8

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}

_PRETOK_RE = _re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)

            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]

        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None

        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]

        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

# It takes head_dim (size of one head), max_len (max length), device and base (rotary base value) as input, pre-computes the sine and cosine tables for rotary positions and returns (cos table, sin table).
def build_rope_cache(head_dim, max_len, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

# It takes x (input tensor) as input, rotates half of the vector, used by rotary positions and returns rotated tensor.
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

# It takes q (queries), k (keys), cos (rotary cos table), sin (rotary sin table) and pos_offset (position offset) as input, adds rotary position information to queries and keys and returns (rotated queries, rotated keys).
def apply_rope(q, k, cos, sin, pos_offset):
    T = q.size(2)
    cos_t = cos[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    sin_t = sin[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    q_rot = q * cos_t + rotate_half(q) * sin_t
    k_rot = k * cos_t + rotate_half(k) * sin_t
    return q_rot, k_rot

USE_SDPA = True

class CausalSelfAttention(nn.Module):

    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates the attention layer and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, dim)
        self.wk = nn.Linear(dim, dim)
        self.wv = nn.Linear(dim, dim)
        self.wo = nn.Linear(dim, dim)
        self.attn_drop_p = dropout
        self.resid_drop = nn.Dropout(dropout)
        self.bidirectional = False
        self.key_pad_mask = None

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs attention where each token looks only at earlier tokens and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin, pos_offset)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)

        Tq, Tk = q.size(2), k.size(2)
        q_idx = torch.arange(Tk - Tq, Tk, device=x.device)
        k_idx = torch.arange(Tk, device=x.device)
        causal_mask = k_idx.unsqueeze(0) > q_idx.unsqueeze(1)
        if self.bidirectional:
            causal_mask = torch.zeros_like(causal_mask)

        if USE_SDPA:
            allowed = ~causal_mask
            if self.key_pad_mask is not None:
                allowed = allowed[None, None] & self.key_pad_mask[:, None, None, :]
                allowed = allowed | (k_idx.unsqueeze(0) == q_idx.unsqueeze(1))[None, None]
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.attn_drop_p if self.training else 0.0)
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            att = att.masked_fill(causal_mask, float("-inf"))
            if self.key_pad_mask is not None:
                att = att.masked_fill(~self.key_pad_mask[:, None, None, :], -1e9)
            att = F.softmax(att, dim=-1)
            att = F.dropout(att, p=self.attn_drop_p, training=self.training)
            out = att @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.wo(out)), new_cache

class TransformerMLP(nn.Module):
    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * mult)
        self.fc2 = nn.Linear(dim * mult, dim)
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = TransformerMLP(dim, dropout=dropout)

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs one Transformer block and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        a, new_cache = self.attn(self.ln1(x), cos, sin, pos_offset, cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, new_cache

class LegalTransformerLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), n_heads (number of heads), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full Legal Transformer model and returns nothing.
    def __init__(self, vocab_size, dim=896, n_layers=24, n_heads=14, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads, dropout) for _ in range(n_layers)])
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:
            self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope_cache(self.head_dim, max_len, device=torch.device("cpu"))
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            nn.init.normal_(blk.attn.wo.weight, mean=0.0, std=0.02 * residual_scale)
            nn.init.normal_(blk.mlp.fc2.weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), kv_cache (saved keys and values) and pos_offset (position offset) as input, runs the model and returns next-token scores and the cache.
    def forward(self, input_ids, kv_cache=None, pos_offset=0):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, lc = blk(x, self.rope_cos, self.rope_sin, pos_offset, layer_cache)
            new_cache.append(lc)
        logits = self.lm_head(self.norm_f(x))
        return logits, new_cache

class LegalTokenizer(LegalByteBPETokenizer):

    # It returns the vocabulary size.
    @property
    def vocab_size(self):
        return len(self.vocab)

    # It returns the padding token id.
    @property
    def pad_id(self):
        return self.pad_token_id

    # It returns the end-of-text token id.
    @property
    def eos_id(self):
        return self.eos_token_id

    bos_id = None

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):
        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            chunk = m.group()
            base = m.start()
            b2c = []
            for ci, ch in enumerate(chunk):
                b2c.extend([base + ci] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[b] for b in chunk.encode("utf-8"))
            pos = 0
            for piece in self._bpe(sym):
                nb = len(piece)
                ids.append(self.vocab[piece])
                offs.append((b2c[pos], b2c[pos + nb - 1] + 1))
                pos += nb
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return LegalTokenizer.load(tok_path)

# It takes path (file path) as input, finds the real checkpoint file to load and returns (file path, temporary folder or None).
def _resolve_ckpt(path):
    if os.path.isfile(path):
        return path, None
    if not os.path.isdir(path):
        raise FileNotFoundError(path)
    root = None
    for dp, _dn, fn in os.walk(path):
        if "data.pkl" in fn:
            root = dp
            break
    if root is None:
        raise FileNotFoundError(f"{path} is a folder without data.pkl")
    prefix = os.path.splitext(os.path.basename(os.path.normpath(root)))[0] or "archive"
    tmpdir = tempfile.mkdtemp(prefix="legal_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for dp, _dn, fn in os.walk(root):
            for f in sorted(fn):
                full = os.path.join(dp, f)
                z.write(full, arcname=prefix + "/" + os.path.relpath(full, root).replace(os.sep, "/"))
    return out, tmpdir

class Backbone:

    # It takes model, cfg (settings) and rope_len (rotary table length) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg, rope_len):
        self.model = model
        self.config = dict(cfg)
        self.d_model = int(cfg["dim"])
        self.max_len = TRAINED_SEQ_TOKENS
        self.rope_len = rope_len
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False
        self.causal = True

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag: bool):
        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional
        for blk in self.model.blocks:
            blk.attn.bidirectional = self.bidirectional

    # It takes input_ids (token ids) and attention_mask (mask of real tokens) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None):
        m = self.model
        B, T = input_ids.shape
        if T > self.rope_len:
            raise ValueError(f"sequence length {T} > RoPE table {self.rope_len}; "
                             f"load_backbone(..., max_len>={T}) (trained only up to {self.max_len})")
        kpm = None
        if attention_mask is not None:
            am = attention_mask.to(input_ids.device).bool()
            if not bool(am.all()):
                kpm = am
        attns = [blk.attn for blk in m.blocks]
        for a in attns:
            a.key_pad_mask = kpm
        try:
            x = m.drop(m.tok_emb(input_ids))
            for blk in m.blocks:
                x, _ = blk(x, m.rope_cos, m.rope_sin, 0, None)
            return m.norm_f(x)
        finally:
            for a in attns:
                a.key_pad_mask = None

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):
        return [n for n, mod in self.model.named_modules()
                if n.startswith("blocks.") and isinstance(mod, nn.Linear)]

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

# It takes ckpt_path (checkpoint path), device, max_len (max length) and config (model settings) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", max_len=None, config=None):
    path, tmp = _resolve_ckpt(ckpt_path)
    try:
        pkg = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    if isinstance(pkg, dict) and "state_dict" in pkg:
        sd, cfg = pkg["state_dict"], dict(pkg.get("config") or {})
    else:
        sd, cfg = pkg, {}
    if config:
        cfg.update(config)
    sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in sd.items()}
    if "vocab_size" not in cfg or "dim" not in cfg:
        w = sd["tok_emb.weight"]
        cfg.setdefault("vocab_size", w.shape[0]); cfg.setdefault("dim", w.shape[1])
        cfg.setdefault("n_layers", 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks.")))
    for k in ("n_heads", "n_layers"):
        if k not in cfg:
            raise ValueError(f"checkpoint config lacks {k!r}; pass config={{'{k}': ...}}")
    cfg.setdefault("dropout", 0.1)
    cfg.setdefault("tied", "lm_head.weight" not in sd or sd["lm_head.weight"].data_ptr() == sd["tok_emb.weight"].data_ptr())
    rope_len = max(int(cfg.get("max_len", NOTEBOOK_ROPE_LEN)), int(max_len or 0))
    model = LegalTransformerLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                               n_heads=cfg["n_heads"], max_len=rope_len,
                               dropout=cfg["dropout"], tie=cfg["tied"])
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if cfg["tied"]:
        missing = [k for k in missing if k != "lm_head.weight"]
    if missing or unexpected:
        print("!" * 70, f"\n!!! STATE_DICT MISMATCH  missing={missing[:10]} (n={len(missing)})  "
              f"unexpected={unexpected[:10]} (n={len(unexpected)})\n" + "!" * 70)
        raise RuntimeError("checkpoint does not match Legal-Transformer architecture")
    if cfg["tied"]:
        assert model.lm_head.weight is model.tok_emb.weight
    model.to(device)
    bb = Backbone(model, cfg, rope_len)
    bb.max_len = min(TRAINED_SEQ_TOKENS, rope_len)
    return bb

Writing arch_legal_transformer.py


In [4]:
%%writefile ner_core.py
# Main NER code: data, labels, training, scoring (saved as ner_core.py)
import os, re, json, glob, math, time, random, zipfile, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
T2I = {t: i for i, t in enumerate(TAGS)}
NT = len(TAGS)
RARE = ("WIT", "A.COUNSEL", "R.COUNSEL", "APP", "RESP")
WORD_RE = re.compile(r"[^\W\d_]+|\d+|[^\w\s]|_")

# It takes name and roots (folders to search) as input, searches the input folders for a file and returns a file path (or None if not found).
def find_file(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", name), recursive=True)
        hits = [h for h in hits if "__MACOSX" not in h]
        if hits:
            return sorted(hits, key=len)[0]
    return None

# It loads the training documents and their entity spans and returns a dictionary of documents with text and spans.
def load_train_docs():
    docs = {}
    for f in (1, 2):
        tp, gp = find_file(f"fold{f}-text.json"), find_file(f"fold{f}-gold.json")
        assert tp and gp, (f"fold{f}-text.json / fold{f}-gold.json not found under /kaggle/input - "
                           "attach the dataset that contains the Phase-2 Train files")
        T, G = json.load(open(tp)), json.load(open(gp))
        assert set(T) == set(G), f"fold{f}: text ids and gold ids differ"
        for k in T:
            assert k not in docs, f"duplicate doc id {k} across folds"
            sp = [(int(a), int(b), l) for a, b, l, *_ in G[k]]
            for s, e, l in sp:
                assert l in LABELS and 0 <= s < e <= len(T[k]), (k, s, e, l)
            docs[k] = {"text": T[k], "spans": sorted(sp), "fold": f}
    return docs

# It loads the test documents and returns a dictionary of test documents.
def load_test_docs():
    p = find_file("fold3-text.json")
    if not p:
        return None
    T = json.load(open(p))
    return {k: {"text": v, "spans": []} for k, v in T.items()}

# It takes docs (documents), n_val (number of hold-out documents), seed (random seed) and tries (number of random tries) as input, splits documents into train and hold-out, keeping rare labels in both and returns (train ids, hold-out ids).
def make_split(docs, n_val=14, seed=0, tries=4000):
    ids = sorted(docs)
    cnt = {k: np.array([sum(1 for s in docs[k]["spans"] if s[2] == l) for l in LABELS]) for k in ids}
    rng = random.Random(seed)
    best, bscore = None, -1
    tot = sum(cnt.values())
    for _ in range(tries):
        v = rng.sample(ids, n_val)
        c = sum(cnt[k] for k in v)
        score = (c / np.maximum(tot, 1)).min() - 0.5 * abs((c / np.maximum(tot, 1)).mean() - n_val / len(ids))
        if score > bscore:
            best, bscore = v, score
    val = sorted(best)
    return [k for k in ids if k not in set(val)], val

# It takes text as input, splits text into small words and remember their positions and returns (word start positions, word end positions).
def segment(text):
    m = [(x.start(), x.end()) for x in WORD_RE.finditer(text)]
    ws = np.array([a for a, _ in m], dtype=np.int64)
    we = np.array([b for _, b in m], dtype=np.int64)
    return ws, we

# It takes ws (word start positions), we (word end positions) and spans (entity spans) as input, gives each word its B/I/O tag and returns a list of tag ids.
def tag_words(ws, we, spans):
    tags = np.zeros(len(ws), dtype=np.int64)
    for s, e, l in spans:
        i0 = int(np.searchsorted(we, s, side="right"))
        i1 = int(np.searchsorted(ws, e, side="left"))
        if i1 <= i0:
            continue
        tags[i0] = T2I["B-" + l]
        tags[i0 + 1:i1] = T2I["I-" + l]
    return tags

# It takes tag_ids (tag ids), ws (word start positions) and we (word end positions) as input, turns word tags back into entity spans and returns a list of (start, end, label).
def spans_from_tags(tag_ids, ws, we):
    out, cur = [], None
    for i, t in enumerate(tag_ids):
        tg = TAGS[t]
        if tg == "O":
            if cur: out.append(cur); cur = None
        elif tg[0] == "B":
            if cur: out.append(cur)
            cur = [int(ws[i]), int(we[i]), tg[2:]]
        else:
            if cur and cur[2] == tg[2:]:
                cur[1] = int(we[i])
            else:
                if cur: out.append(cur)
                cur = [int(ws[i]), int(we[i]), tg[2:]]
    if cur: out.append(cur)
    return [tuple(x) for x in out]

_TRANS = None

# It builds the table of allowed tag-to-tag moves and returns (allowed move table, allowed start tags).
def _trans():
    global _TRANS
    if _TRANS is None:
        m = np.zeros((NT, NT), dtype=np.float32)
        for b, tb in enumerate(TAGS):
            if tb.startswith("I-"):
                for a, ta in enumerate(TAGS):
                    if ta[2:] != tb[2:] or ta == "O":
                        m[a, b] = -1e4
        start = np.array([-1e4 if t.startswith("I-") else 0.0 for t in TAGS], dtype=np.float32)
        _TRANS = (m, start)
    return _TRANS

# It takes logp (log probabilities) as input, finds the best tag sequence that follows the B/I/O rules and returns the best label id for each position.
def viterbi(logp):
    n = logp.shape[0]
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    tr, start = _trans()
    score = logp[0] + start
    back = np.zeros((n, NT), dtype=np.int64)
    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

# It takes logp (log probabilities), ws (word start positions), we (word end positions) and mode as input, turns predicted tags into entity spans and returns a list of entity spans.
def decode(logp, ws, we, mode="viterbi"):
    path = viterbi(logp) if mode == "viterbi" else logp.argmax(1)
    return spans_from_tags(path, ws, we)

# It takes gold (true labels) and pred (predictions) as input, computes strict precision, recall and F1 for entity spans and returns a dictionary of scores (overall and per label).
def strict_scores(gold, pred):
    tp = {l: 0 for l in LABELS}; fp = dict(tp); fn = dict(tp)
    for d in gold:
        g = {(int(a), int(b), l) for a, b, l, *_ in gold[d]}
        p = {(int(a), int(b), l) for a, b, l, *_ in pred.get(d, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in LABELS:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = dict(P=P, R=R, F1=(2 * P * R / (P + R) if P + R else 0.0), tp=tp[l], fp=fp[l], fn=fn[l])
    out = {"per_label": per,
           "macro_P": float(np.mean([per[l]["P"] for l in LABELS])),
           "macro_R": float(np.mean([per[l]["R"] for l in LABELS])),
           "macro_F1": float(np.mean([per[l]["F1"] for l in LABELS]))}
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mp, mr = T / max(1, T + FP), T / max(1, T + FN)
    out["micro_F1"] = 2 * mp * mr / max(1e-9, mp + mr)
    return out

# It takes sc (scores) and title as input, prints the scores for each label and returns nothing.
def print_scores(sc, title=""):
    print(title, "strict macro-F1 %.4f  (P %.4f  R %.4f)  micro-F1 %.4f" % (sc["macro_F1"], sc["macro_P"], sc["macro_R"], sc["micro_F1"]))
    for l in LABELS:
        r = sc["per_label"][l]
        print("   %-10s F1 %.3f  P %.3f  R %.3f   tp %4d fp %4d fn %4d" % (l, r["F1"], r["P"], r["R"], r["tp"], r["fp"], r["fn"]))

# It takes docs (documents) and ids (document ids) as input, checks the best score possible with this word splitting and returns scores of the rebuilt gold spans.
def oracle_check(docs, ids):
    pred, gold = {}, {}
    for k in ids:
        ws, we = segment(docs[k]["text"])
        pred[k] = spans_from_tags(tag_words(ws, we, docs[k]["spans"]), ws, we)
        gold[k] = docs[k]["spans"]
    return strict_scores(gold, pred)

class Enc:

    # It takes doc_id (document id), text, tok (tokenizer), spans (entity spans) and soft (teacher probabilities) as input, tokenizes a document and line up its words and tags and returns nothing.
    def __init__(self, doc_id, text, tok, spans=None, soft=None):
        self.id, self.text, self.spans = doc_id, text, spans or []
        ids, offs = tok.encode_with_offsets(text)
        self.ids = np.asarray(ids, dtype=np.int64)
        offs = np.asarray(offs, dtype=np.int64).reshape(-1, 2)
        self.ws, self.we = segment(text)
        starts, ends = offs[:, 0], offs[:, 1]
        n = len(self.ids)
        self.wl = np.clip(np.searchsorted(starts, self.we, side="left") - 1, 0, n - 1)
        self.wf = np.clip(np.searchsorted(ends, self.ws, side="right"), 0, n - 1)
        self.wf = np.minimum(self.wf, self.wl)
        self.tags = tag_words(self.ws, self.we, self.spans) if spans is not None else None
        self.soft = soft

    # It returns the number of words.
    @property
    def n_words(self):
        return len(self.ws)

# It takes docs (documents), ids (document ids) and tok (tokenizer) as input, tokenizes all documents and returns a list of encoded documents.
def build_encs(docs, ids, tok):
    t0 = time.time()
    out = [Enc(k, docs[k]["text"], tok, docs[k].get("spans", [])) for k in ids]
    print(f"encoded {len(out)} docs, {sum(len(e.ids) for e in out):,} tokens, {sum(e.n_words for e in out):,} words in {time.time()-t0:.0f}s")
    return out

# It takes encs (encoded documents) as input, shows simple token length statistics and returns a dictionary with tokens per character.
def token_stats(encs):
    r = np.mean([len(e.ids) / max(1, len(e.text)) for e in encs])
    return dict(tokens_per_char=float(r), chars_per_token=float(1 / r))

# It takes enc (encoded document), W (window size), stride (window stride) and shift (window shift) as input, cuts a document into overlapping token windows and returns a list of (start, end) windows.
def windows_for(enc, W, stride, shift=0):
    n = len(enc.ids)
    if n <= W:
        return [(0, n)]
    w, a = [], shift % stride if shift else 0
    if a: w.append((0, min(W, n)))
    while True:
        b = min(a + W, n)
        w.append((a, b))
        if b >= n: break
        a += stride
    return w

# It takes enc (encoded document), a (window start) and b (window end) as input, finds the words that lie fully inside a window and returns indices of the words in the window.
def window_words(enc, a, b):
    m = (enc.wf >= a) & (enc.wl < b)
    return np.nonzero(m)[0]

CTX_TOKENS = 0

# It takes items (list of items), double (read text twice), sep_id (separator id), pad_id (padding id) and device as input, builds padded tensors for one batch and returns a dictionary of tensors.
def make_batch(items, double, sep_id, pad_id, device):
    H = CTX_TOKENS
    assert not (H and double), "header context and double-pass cannot be combined"
    L = max((b - a) for _, a, b, _ in items) + (H + 1 if H else 0)
    TL = (2 * L + 1) if double else L
    Wm = max(1, max(len(wi) for *_, wi in items))
    B = len(items)
    ids = np.full((B, TL), pad_id, dtype=np.int64)
    widx = np.zeros((B, Wm), dtype=np.int64)
    wmask = np.zeros((B, Wm), dtype=bool)
    tags = np.full((B, Wm), -100, dtype=np.int64)
    soft = None
    if any(e.soft is not None for e, *_ in items):
        soft = np.zeros((B, Wm, NT), dtype=np.float32)
    for i, (e, a, b, wi) in enumerate(items):
        off = 0
        if H and a > 0:
            if a <= H:
                seg = e.ids[:b]
                n = len(seg); ids[i, :n] = seg
                k = len(wi); widx[i, :k] = e.wl[wi]; wmask[i, :k] = True
                if e.tags is not None: tags[i, :k] = e.tags[wi]
                if soft is not None and e.soft is not None: soft[i, :k] = e.soft[wi]
                continue
            seg = np.concatenate([e.ids[:H], [sep_id], e.ids[a:b]]); n = len(seg)
            ids[i, :n] = seg
            off = H + 1
        else:
            seg = e.ids[a:b]; n = len(seg)
            ids[i, :n] = seg
        if double:
            ids[i, n] = sep_id
            ids[i, n + 1:2 * n + 1] = seg
            off = n + 1
        k = len(wi)
        widx[i, :k] = e.wl[wi] - a + off
        wmask[i, :k] = True
        if e.tags is not None: tags[i, :k] = e.tags[wi]
        if soft is not None and e.soft is not None: soft[i, :k] = e.soft[wi]
    t = lambda x: torch.from_numpy(x).to(device)
    out = dict(ids=t(ids), widx=t(widx), wmask=t(wmask), tags=t(tags))
    if soft is not None: out["soft"] = t(soft)
    return out

# It takes encs (encoded documents), W (window size), stride (window stride), epoch_seed (seed for this epoch), rare_dup (copies of rare windows) and min_words (minimum words) as input, makes the list of training windows for one epoch and returns a list of training windows.
def train_items(encs, W, stride, epoch_seed, rare_dup=1, min_words=1):
    rng = random.Random(epoch_seed)
    items = []
    rare_ids = {T2I["B-" + l] for l in RARE}
    for e in encs:
        for a, b in windows_for(e, W, stride, shift=rng.randrange(stride)):
            wi = window_words(e, a, b)
            if len(wi) < min_words: continue
            items.append((e, a, b, wi))
            if rare_dup and e.tags is not None and any(int(t) in rare_ids for t in e.tags[wi]):
                items.extend([(e, a, b, wi)] * rare_dup)
    rng.shuffle(items)
    return items

class NERModel(nn.Module):

    # It takes bb (pre-trained backbone), gru_hidden (GRU size), dropout (dropout rate), pool (pooling type), double (read text twice), use_gru (use a GRU) and sep_id (separator id) as input, builds the tagging model: backbone plus BiGRU and tag layer and returns nothing.
    def __init__(self, bb, gru_hidden=256, dropout=0.2, pool="last", double=False, use_gru=True, sep_id=1):
        super().__init__()
        self.bb_model = bb.model
        self.bb = bb
        self.pool, self.double, self.use_gru, self.sep_id = pool, double, use_gru, sep_id
        d = bb.d_model
        self.drop = nn.Dropout(dropout)
        if use_gru:
            self.gru = nn.GRU(d, gru_hidden, batch_first=True, bidirectional=True)
            dout = 2 * gru_hidden
        else:
            dout = d
        self.out = nn.Linear(dout, NT)

    # It takes ids (token ids), widx (token index of each word) and wmask (word mask) as input, runs the model and returns tag scores for each word.
    def forward(self, ids, widx, wmask):
        shim = copy.copy(self.bb); shim.model = self.bb_model
        h = shim.hidden(ids)
        idx = widx.unsqueeze(-1).expand(-1, -1, h.size(-1))
        x = torch.gather(h, 1, idx)
        x = self.drop(x.float())
        if self.use_gru:
            lens = wmask.sum(1).clamp(min=1).cpu()
            pk = nn.utils.rnn.pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False)
            y, _ = self.gru(pk)
            x, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=widx.size(1))
            x = self.drop(x)
        return self.out(x)

# It takes model as input, returns a list of weight names; it is the names of the weights that belong to the new head.
def head_param_names(model):
    return [n for n, _ in model.named_parameters() if not n.startswith("bb_model.")]

# It takes model as input, counts trainable and total parameters and returns (trainable parameters, total parameters).
def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    al = sum(p.numel() for p in model.parameters())
    return tr, al

# It takes e (error) as input, checks whether an error is a GPU out-of-memory error and returns True or False.
def _is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

# It takes model, encs (encoded documents), W (window size), stride (window stride), bs (batch size), double (read text twice), pad_id (padding id), sep_id (separator id), device and use_amp (use mixed precision) as input, predicts probabilities, averaging overlapping windows and returns probabilities for every document.
@torch.no_grad()
def predict_probs(model, encs, W, stride, bs=8, double=False, pad_id=0, sep_id=1, device="cuda", use_amp=True):
    model.eval()
    outs = [np.zeros((e.n_words, NT), dtype=np.float32) for e in encs]
    wsum = [np.zeros(e.n_words, dtype=np.float32) for e in encs]
    items = []
    for di, e in enumerate(encs):
        for a, b in windows_for(e, W, stride):
            wi = window_words(e, a, b)
            if len(wi): items.append((di, e, a, b, wi))
    items.sort(key=lambda x: len(x[1].ids[x[2]:x[3]]))
    i, cur = 0, bs
    while i < len(items):
        chunk = items[i:i + cur]
        batch = make_batch([(e, a, b, wi) for _, e, a, b, wi in chunk], double, sep_id, pad_id, device)
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp and device != "cpu"):
                lg = model(batch["ids"], batch["widx"], batch["wmask"])
        except RuntimeError as ex:
            if _is_oom(ex) and cur > 1:
                torch.cuda.empty_cache(); cur = max(1, cur // 2); print(f"  OOM in inference -> batch {cur}"); continue
            raise
        pr = torch.softmax(lg.float(), -1).cpu().numpy()
        for j, (di, e, a, b, wi) in enumerate(chunk):
            k = len(wi)
            pos = np.arange(k)
            w = (np.minimum(pos, k - 1 - pos) + 1).astype(np.float32)
            outs[di][wi] += pr[j, :k] * w[:, None]
            wsum[di][wi] += w
        i += len(chunk)
    for o, s in zip(outs, wsum):
        miss = s == 0
        o[~miss] /= s[~miss][:, None]
        if miss.any(): o[miss, 0] = 1.0
    return outs

# It takes encs (encoded documents), probs (probabilities) and mode as input, turns tag probabilities into entity spans and returns predicted spans per document.
def probs_to_spans(encs, probs, mode="viterbi"):
    return {e.id: decode(np.log(np.clip(p, 1e-9, 1)), e.ws, e.we, mode) for e, p in zip(encs, probs)}

# It takes model, encs (encoded documents), W (window size), stride (window stride), bs (batch size), double (read text twice), pad_id (padding id), sep_id (separator id), device and mode as input, predicts on documents and computes the scores and returns (scores, probabilities, predictions).
def evaluate(model, encs, W, stride, bs=8, double=False, pad_id=0, sep_id=1, device="cuda", mode="viterbi"):
    probs = predict_probs(model, encs, W, stride, bs, double, pad_id, sep_id, device)
    pred = probs_to_spans(encs, probs, mode)
    gold = {e.id: e.spans for e in encs}
    return strict_scores(gold, pred), probs, pred

# It takes encs (encoded documents), power and cap (maximum value) as input, gives rare tags a higher loss weight and returns a tensor of tag weights.
def tag_class_weights(encs, power=0.5, cap=6.0):
    cnt = np.ones(NT)
    for e in encs:
        cnt += np.bincount(e.tags[e.tags >= 0], minlength=NT)
    ent = cnt[1:]
    w = np.ones(NT)
    ref = np.median(ent)
    w[1:] = np.clip((ref / ent) ** power, 0.5, cap)
    return torch.tensor(w, dtype=torch.float32)

# It takes model as input, copies the trainable weights and returns a dictionary of trainable weights.
def _tstate(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if _trainable_key(model, k)}

PROGRESS = dict(total=0.0, done=0.0, skipped=0.0, t0=None)

# It takes sec (seconds) as input, formats seconds as hours:minutes:seconds and returns time as text.
def _hms(sec):
    sec = int(max(0, sec)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

# It shows overall progress and time left and returns progress text.
def progress_line():
    P = PROGRESS
    if not P["total"] or not P["t0"] or P["done"] <= 0: return ""
    el = time.time() - P["t0"]; rate = P["done"] / max(1e-9, el); left = max(0.0, P["total"] - P["done"] - P["skipped"])
    return f"[overall {100 * min(1.0, (P['done'] + P['skipped']) / P['total']):.0f}% | elapsed {_hms(el)} | ETA {_hms(left / rate)}]"

# It takes model, train_encs (encoded training documents), val_encs (encoded validation documents), cfg (settings), param_groups (parameter groups), device and log (print function) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(model, train_encs, val_encs, cfg, param_groups, device="cuda", log=print):
    use_amp = cfg.get("amp", True) and device != "cpu"
    opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.98), weight_decay=cfg.get("wd", 0.01))
    base_lr = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = tag_class_weights(train_encs, cfg.get("class_power", 0.5)).to(device)
    dmodel = nn.DataParallel(model) if (device == "cuda" and torch.cuda.device_count() > 1) else model
    hint = cfg.get("bs_state")
    bs, accum = (min(cfg["bs"], hint["bs"]) if hint else cfg["bs"]), cfg["accum"]
    eff = cfg["bs"] * accum
    n_items = len(train_items(train_encs, cfg["W"], cfg["stride"], 0, cfg.get("rare_dup", 1)))
    steps_per_epoch = max(1, n_items // eff)
    total = steps_per_epoch * cfg["epochs"]
    warm = max(1, int(cfg.get("warmup", 0.06) * total))
    log(f"windows/epoch {n_items}, optimizer steps/epoch {steps_per_epoch}, total {total}, micro-batch {bs} x accum {accum}")
    step, best, best_ep, bad, hist, best_state = 0, -1.0, -1, 0, [], None
    avg_state, avg_n, ep0 = None, 0, 0
    kd_a, kd_T = cfg.get("kd_alpha", 0.0), cfg.get("kd_T", 2.0)
    train_params = [p for g in param_groups for p in g["params"]]
    patience = cfg.get("patience", 4)
    swa_last, snaps = int(cfg.get("swa_last", 0)), set(cfg.get("snapshot_epochs") or ())
    ckpt, save_every = cfg.get("ckpt_path"), float(cfg.get("save_every", 1800))
    if ckpt and os.path.exists(ckpt):
        try:
            ck = torch.load(ckpt, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["state"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
            step, ep0, best, best_ep, bad, hist = ck["step"], ck["ep"], ck["best"], ck["best_ep"], ck["bad"], ck["hist"]
            best_state, avg_state, avg_n = ck["best_state"], ck["avg_state"], ck["avg_n"]
            log(f"  resumed from {os.path.basename(ckpt)} at epoch {ep0 + 1}/{cfg['epochs']}"); del ck
            PROGRESS["skipped"] += ep0 * len(train_encs)
        except Exception as ex:
            log(f"  could not resume from {ckpt} ({str(ex)[:100]}) -> training from scratch"); step, ep0, best, best_ep, bad, hist, best_state, avg_state, avg_n = 0, 0, -1.0, -1, 0, [], None, None, 0
    t_save = time.time()
    for ep in range(ep0, cfg["epochs"]):
        model.train()
        items = train_items(train_encs, cfg["W"], cfg["stride"], cfg.get("seed", 0) * 1000 + ep, cfg.get("rare_dup", 1))
        t0, run, nrun = time.time(), torch.zeros((), device=device), 0
        if PROGRESS["t0"] is None: PROGRESS["t0"] = t0
        try:
            from tqdm.auto import tqdm as _tqdm
            bar = _tqdm(total=steps_per_epoch, desc=f"epoch {ep + 1}/{cfg['epochs']}", mininterval=20, leave=False, dynamic_ncols=True)
        except Exception:
            bar = None
        i = 0
        while i + eff <= len(items) or (i == 0 and len(items)):
            lr_f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            lr_f = max(lr_f, 0.02) if step >= warm else lr_f
            for g, b0 in zip(opt.param_groups, base_lr): g["lr"] = b0 * lr_f
            micro = items[i:i + eff]; i += eff
            j = 0
            while j < len(micro):
                part = micro[j:j + bs]
                batch = make_batch(part, cfg.get("double", False), cfg.get("sep_id", 1), cfg.get("pad_id", 0), device)
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        lg = dmodel(batch["ids"], batch["widx"], batch["wmask"]).float()
                    mask = batch["tags"] != -100
                    ce = F.cross_entropy(lg[mask], batch["tags"][mask], weight=cw, label_smoothing=cfg.get("label_smooth", 0.0))
                    loss = ce
                    if kd_a > 0 and "soft" in batch:
                        lp = F.log_softmax(lg[mask] / kd_T, -1)
                        tgt = batch["soft"][mask]
                        tgt = tgt.clamp_min(1e-9) ** (1 / kd_T); tgt = tgt / tgt.sum(-1, keepdim=True)
                        kd = F.kl_div(lp, tgt, reduction="batchmean") * kd_T * kd_T
                        loss = (1 - kd_a) * ce + kd_a * kd
                    scaler.scale(loss * len(part) / len(micro)).backward()
                    run += ce.detach().float(); nrun += 1
                    j += len(part)
                except RuntimeError as ex:
                    if dmodel is not model and not _is_oom(ex):
                        log(f"  DataParallel failed ({str(ex)[:120]}) -> continuing on a single GPU")
                        dmodel = model; opt.zero_grad(set_to_none=True); j = 0; continue
                    if _is_oom(ex) and bs > 1:
                        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                        bs = max(1, bs // 2)
                        if hint is not None: hint["bs"] = bs
                        log(f"  OOM -> micro-batch {bs}"); j = 0; continue
                    raise
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(train_params, cfg.get("clip", 1.0))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if bar is not None:
                bar.update(1)
                if step % 25 == 0: bar.set_postfix(ce=f"{float(run) / max(1, nrun):.3f}")
        if bar is not None: bar.close()
        PROGRESS["done"] += len(train_encs)
        rec = dict(epoch=ep + 1, train_ce=float(run) / max(1, nrun), secs=time.time() - t0)
        if val_encs and ep + 1 >= cfg.get("eval_from", 1):
            sc, _, _ = evaluate(model, val_encs, cfg["W"], cfg["stride"], bs=max(2, bs * 2), double=cfg.get("double", False),
                                pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
            rec.update(val_macro_F1=sc["macro_F1"], val_micro_F1=sc["micro_F1"])
            if sc["macro_F1"] > best:
                best, best_ep, bad = sc["macro_F1"], ep + 1, 0
                best_state = _tstate(model)
            else:
                bad += 1
        if swa_last and ep + 1 > cfg["epochs"] - swa_last:
            cur = _tstate(model)
            if avg_state is None: avg_state, avg_n = {k: v.float() for k, v in cur.items()}, 1
            else:
                avg_n += 1
                for k, v in cur.items(): avg_state[k] += (v.float() - avg_state[k]) / avg_n
        hist.append(rec)
        eta_run = (time.time() - t0) * (cfg["epochs"] - ep - 1)
        log("  ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()) + f"  | this run: epoch {ep + 1}/{cfg['epochs']}, ETA {_hms(eta_run)}  " + progress_line())
        if (ep + 1) in snaps and cfg.get("snapshot_fn"):
            cfg["snapshot_fn"](ep + 1, model)
        if ckpt and ep + 1 < cfg["epochs"] and time.time() - t_save >= save_every:
            torch.save(dict(state=_tstate(model), opt=opt.state_dict(), scaler=scaler.state_dict(), step=step, ep=ep + 1, best=best, best_ep=best_ep,
                            bad=bad, hist=hist, best_state=best_state, avg_state=avg_state, avg_n=avg_n), ckpt + ".tmp")
            os.replace(ckpt + ".tmp", ckpt); t_save = time.time(); log(f"  [saved resumable checkpoint after epoch {ep + 1}]")
        if val_encs and cfg.get("select", "best") == "best" and patience > 0 and bad >= patience:
            log(f"early stop (no improvement for {bad} evaluations)"); break
    last_state = _tstate(model)
    if not val_encs or cfg.get("select", "best") == "last" or best_state is None:
        best_state, best_ep = last_state, len(hist)
    if cfg.get("return_last"):
        avg_out = {k: v.to(last_state[k].dtype) if last_state[k].is_floating_point() else last_state[k] for k, v in avg_state.items()} if avg_state else None
        return hist, best_ep, best_state, {"last": last_state, "avg": avg_out}
    return hist, best_ep, best_state

# It takes model and k (weight name) as input, checks whether a weight is trainable and returns True or False.
def _trainable_key(model, k):
    sd_params = dict(model.named_parameters())
    return (k in sd_params and sd_params[k].requires_grad)

# It takes model and state (weights) as input, loads the best saved weights into the model and returns nothing.
def load_best(model, state):
    miss = model.load_state_dict(state, strict=False)
    assert not miss.unexpected_keys, miss.unexpected_keys

# It takes pred (predictions), test_docs (test documents), path_zip (zip path) and with_text (include text) as input, checks predictions and writes the submission zip and returns a path of the zip.
def write_submission(pred, test_docs, path_zip, with_text=False):
    assert set(pred) == set(test_docs), f"id mismatch: missing {len(set(test_docs)-set(pred))}, extra {len(set(pred)-set(test_docs))}"
    out = {}
    for k, sp in pred.items():
        n = len(test_docs[k]["text"]); rows = []
        for s, e, l in sorted(sp):
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= n and l in LABELS, (k, s, e, l)
            rows.append([s, e, l, test_docs[k]["text"][s:e]] if with_text else [s, e, l])
        out[k] = rows
    os.makedirs(os.path.dirname(path_zip) or ".", exist_ok=True)
    with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(out))
    n_ent = sum(len(v) for v in out.values())
    print(f"wrote {path_zip}: {len(out)} docs, {n_ent} entities")
    return out

# It takes s (random seed) as input, sets random seeds so runs are repeatable and returns nothing.
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# It takes gold (true labels) and pred (predictions) as input, counts the different kinds of mistakes and returns (mistake counts, mistakes per label).
def error_analysis(gold, pred):
    from collections import Counter
    cnt = Counter(); by = {l: Counter() for l in LABELS}
    for d in gold:
        g = [(int(a), int(b), l) for a, b, l, *_ in gold[d]]
        p = [(int(a), int(b), l) for a, b, l, *_ in pred.get(d, [])]
        gs, ps = set(g), set(p)
        for (s, e, l) in g:
            if (s, e, l) in ps: by[l]["correct"] += 1; continue
            ov = [q for q in p if q[0] < e and s < q[1]]
            if not ov: kind = "missed"
            elif any(q[0] == s and q[1] == e for q in ov): kind = "label"
            elif any(q[2] == l for q in ov): kind = "boundary"
            else: kind = "both"
            by[l][kind] += 1; cnt[kind] += 1
        for (s, e, l) in p:
            if (s, e, l) in gs: continue
            if not any(q[0] < e and s < q[1] for q in g):
                by[l]["spurious"] += 1; cnt["spurious"] += 1
    return cnt, by

# It takes cnt (mistake counts) and by (mistakes per label) as input, prints the mistake counts and returns nothing.
def print_errors(cnt, by):
    print("error types:", dict(cnt))
    print("%-10s %8s %8s %8s %8s %8s %9s" % ("gold label", "correct", "boundary", "label", "both", "missed", "spurious*"))
    for l in LABELS:
        b = by[l]
        print("%-10s %8d %8d %8d %8d %8d %9d" % (l, b["correct"], b["boundary"], b["label"], b["both"], b["missed"], b["spurious"]))
    print("(*spurious = predicted entities of that label that overlap no gold entity)")

# It takes path (file path), encs (encoded documents) and probs (probabilities) as input, saves predicted probabilities to a file and returns nothing.
def save_probs(path, encs, probs):
    np.savez_compressed(path, ids=np.array([e.id for e in encs]), **{f"p_{e.id}": p.astype(np.float16) for e, p in zip(encs, probs)})

# It takes repo (dataset name) and dest (download folder) as input, downloads the Phase-2 data if it is not attached and returns nothing.
def ensure_phase2_data(repo="Exploration-Lab/CS-781-Capstone", dest="/kaggle/working/phase2_data"):
    names = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json", "labels.txt", "fold3-text.json"]
    if all(find_file(n) for n in names[:5]):
        print("Phase-2 files found locally:", os.path.dirname(find_file("fold1-text.json")))
        return
    from huggingface_hub import HfApi, hf_hub_download, login
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("Hugging Face token (read access, dataset terms accepted): ")
    login(token=tok)
    files = HfApi().list_repo_files(repo, repo_type="dataset")
    os.makedirs(dest, exist_ok=True)
    for n in names:
        c = [f for f in files if f == n or f.endswith("/" + n)]
        pref = [f for f in c if any(k in f.lower() for k in ("ner", "phase_2", "phase2"))]
        c = pref or c
        if not c:
            if n == "fold3-text.json":
                print("  (fold3-text.json not found in the repo - test predictions will be skipped)"); continue
            raise RuntimeError(f"{n} not found in {repo}. Files look like: {files[:15]}")
        if len(c) > 1:
            print(f"  note: {n} appears {len(c)} times {c[:4]}; using {sorted(c, key=len)[0]}")
        p = hf_hub_download(repo, sorted(c, key=len)[0], repo_type="dataset", local_dir=dest)
        print("  downloaded", n, "->", p)

# It takes model, encs (encoded documents), W (window size), strides (window strides) and **kw (extra options) as input, averages predictions made with different window overlaps and returns averaged probabilities for every document.
def predict_probs_multi(model, encs, W, strides, **kw):
    acc = None
    for st in strides:
        p = predict_probs(model, encs, W, st, **kw)
        acc = p if acc is None else [a + b for a, b in zip(acc, p)]
    return [a / len(strides) for a in acc]

# It takes docs (documents), k (number of folds), seed (random seed) and tries (number of random tries) as input, splits documents into folds, spreading rare labels evenly and returns a list of folds (lists of document ids).
def make_folds(docs, k, seed=0, tries=3000):
    ids = sorted(docs)
    cnt = {i: np.array([sum(1 for s in docs[i]["spans"] if s[2] == l) for l in LABELS], dtype=np.float64) for i in ids}
    tot = sum(cnt.values()) + 1
    rng = random.Random(seed)
    best, bs = None, 1e18
    for _ in range(tries):
        order = ids[:]; rng.shuffle(order)
        folds = [order[j::k] for j in range(k)]
        c = np.array([sum(cnt[i] for i in f) for f in folds]) / tot
        score = float(((c - 1.0 / k) ** 2).sum(0).max() + 0.2 * ((c - 1.0 / k) ** 2).sum())
        if score < bs: best, bs = folds, score
    return [sorted(f) for f in best]

Writing ner_core.py


In [5]:
%%writefile ner_peft.py
# Fine-tuning methods: LoRA, LoRMA, adapters (saved as ner_peft.py)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# It takes model and name as input, gets a sub-layer of the model by its name and returns the sub-layer.
def _get(model, name):
    m = model
    for p in name.split("."):
        m = getattr(m, p)
    return m

# It takes model, name and new (new layer) as input, replaces a sub-layer of the model by its name and returns nothing.
def _set(model, name, new):
    parts = name.split(".")
    parent = _get(model, ".".join(parts[:-1])) if len(parts) > 1 else model
    setattr(parent, parts[-1], new)

class LoRALinear(nn.Module):
    # It takes base (base linear layer), r (rank), alpha (scale of the low-rank update) and dropout (dropout rate) as input, creates the small LoRA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the linear layer plus the LoRA update and returns the output of the updated layer.
    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))

class LoRMALinear(nn.Module):

    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.s = base, alpha / r
        o = base.out_features
        self.A = nn.Parameter(torch.empty(r, o)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(o, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

# It takes model as input, freezes all weights of the model and returns nothing.
def freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)

# It takes bb (pre-trained backbone), kind (type), names (layer names), r (rank), alpha (scale of the low-rank update) and dropout (dropout rate) as input, freezes the model and wraps chosen linear layers with LoRA or LoRMA and returns a number of layers wrapped.
def apply_linear_wrappers(bb, kind, names, r=8, alpha=16, dropout=0.05):
    freeze(bb.model)
    n = 0
    for name in names:
        base = _get(bb.model, name)
        if not isinstance(base, nn.Linear):
            continue
        new = LoRALinear(base, r, alpha, dropout) if kind == "lora" else LoRMALinear(base, r, alpha)
        _set(bb.model, name, new); n += 1
    assert n > 0, "no Linear layers wrapped - check the name list"
    return n

class Adapter(nn.Module):
    # It takes d (model size), bottleneck (adapter size) and dropout (dropout rate) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up, self.drop = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    # It takes h (hidden vectors) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(self, h):
        return h + self.up(self.drop(F.gelu(self.down(h))))

# It takes bb (pre-trained backbone) as input, returns the list of model blocks.
def get_blocks(bb):
    if hasattr(bb, "block_modules"):
        return list(bb.block_modules())
    return list(bb.model.blocks)

# It takes bb (pre-trained backbone), bottleneck (adapter size), dropout (dropout rate) and every (how often) as input, freezes the model and adds an adapter after the blocks and returns a number of adapters added.
def add_block_adapters(bb, bottleneck=64, dropout=0.1, every=1):
    freeze(bb.model)
    blocks = get_blocks(bb)
    n = 0
    for i, blk in enumerate(blocks):
        if i % every: continue
        blk.add_module("ner_adapter", Adapter(bb.d_model, bottleneck, dropout))

        # It takes mod (module), inp (layer input) and out (block output) as input, adds the adapter output to the block output and returns block output plus adapter output.
        def hook(mod, inp, out):
            ad = mod.ner_adapter
            if isinstance(out, tuple):
                return (ad(out[0]),) + tuple(out[1:])
            return ad(out)
        blk.register_forward_hook(hook); n += 1
    return n

# It takes bb (pre-trained backbone), last_k_blocks (number of top blocks to train), train_norms (also train norm layers) and train_embeddings (also train embeddings) as input, freezes the model except the last few blocks and returns a number of trainable parameters.
def partial_finetune(bb, last_k_blocks=4, train_norms=True, train_embeddings=False):
    freeze(bb.model)
    blocks = get_blocks(bb)
    for blk in blocks[-last_k_blocks:]:
        for p in blk.parameters(): p.requires_grad_(True)
    if train_norms:
        for n, p in bb.model.named_parameters():
            if "norm_f" in n or n.startswith("norm.") or n.startswith("ln_f"):
                p.requires_grad_(True)
    if train_embeddings:
        for n, p in bb.model.named_parameters():
            if "embed" in n or "tok" in n.split(".")[0]: p.requires_grad_(True)
    return sum(p.numel() for p in bb.model.parameters() if p.requires_grad)

# It takes bb (pre-trained backbone), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, gives lower layers a smaller learning rate than top layers and returns a list of parameter groups with learning rates.
def layerwise_lr_groups(bb, lr_top, decay=0.85, head_params=(), head_lr=1e-3, wd=0.01):
    blocks = get_blocks(bb)
    n = len(blocks)
    seen, groups = set(), []
    for i, blk in enumerate(blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        for p in ps: seen.add(id(p))
        groups.append(dict(params=ps, lr=lr_top * decay ** (n - 1 - i), weight_decay=wd))
    rest = [p for p in bb.model.parameters() if p.requires_grad and id(p) not in seen]
    groups.append(dict(params=rest, lr=lr_top * decay ** n, weight_decay=wd))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return [g for g in groups if g["params"]]

Writing ner_peft.py


In [6]:
%%writefile ner_decode.py
# Decoding options for predicted tags (saved as ner_decode.py)
import numpy as np
import ner_core as C

VARIANTS = {"viterbi": {}, "ob-0.5": dict(ob=-0.5), "ob-1.0": dict(ob=-1.0), "ob-1.5": dict(ob=-1.5), "topk2": dict(topk=2), "topp0.9": dict(topp=0.9)}

# It takes p (probabilities), ob (no-entity penalty), topk (top-k), topp (top-p) and lb (label bias) as input, turns probabilities into log scores with the chosen decoding options and returns log scores.
def _logp(p, ob=0.0, topk=0, topp=0.0, lb=None):
    q = np.clip(p, 1e-9, 1).astype(np.float64)
    if topk:
        drop = np.argsort(-q, 1)[:, topk:]; q = q.copy(); np.put_along_axis(q, drop, 1e-9, 1)
    if topp:
        o = np.argsort(-q, 1); sp = np.take_along_axis(q, o, 1); keep = (np.cumsum(sp, 1) - sp) < topp
        m = np.zeros(q.shape, bool); np.put_along_axis(m, o, keep, 1); q = np.where(m, q, 1e-9)
    lq = np.log(np.clip(q / q.sum(1, keepdims=True), 1e-9, 1)); lq[:, 0] += ob
    if lb is not None:
        for j, l in enumerate(C.LABELS):
            if lb[j]: lq[:, C.T2I["B-" + l]] += lb[j]; lq[:, C.T2I["I-" + l]] += lb[j]
    return lq

# It takes p (probabilities), ws (word start positions), we (word end positions) and **kw (extra options) as input, decodes one document into entity spans and returns a list of spans.
def decode_doc(p, ws, we, **kw):
    return C.decode(_logp(p, **kw), ws, we)

# It takes probs (probabilities), segs (word positions) and **kw (extra options) as input, decodes all documents into entity spans and returns spans per document.
def decode_all(probs, segs, **kw):
    return {i: decode_doc(p, *segs[i], **kw) for i, p in probs.items()}

# It takes probs (probabilities), golds (true spans), segs (word positions) and variants (decoding options) as input, scores every decoding option on the hold-out set and returns a dictionary from decoding option to macro F1.
def score_variants(probs, golds, segs, variants=None):
    out = {}
    for n, kw in (variants or VARIANTS).items():
        out[n] = C.strict_scores(golds, decode_all(probs, segs, **kw))["macro_F1"]
    return out

Writing ner_decode.py


In [7]:
%%writefile ner_opt.py
# Long training, saving and ensembling (saved as ner_opt.py)
import os, json, time, shutil, glob, gc, zipfile
import numpy as np, torch
import ner_core as C, ner_decode as D

WEIGHTS = ("best", "final", "avg")

class Work:
    # It takes out (output folder), resume_from (folder of an earlier run), save_every (save interval (seconds)) and log (print function) as input, sets up the work folder and loads saved progress and returns nothing.
    def __init__(self, out, resume_from=None, save_every=1800, log=print):
        self.out = out; self.dir = os.path.join(out, "work"); os.makedirs(self.dir, exist_ok=True)
        os.makedirs(os.path.join(out, "submissions"), exist_ok=True)
        self.log, self.save_every = log, save_every
        if resume_from and os.path.isdir(resume_from):
            n = 0
            for root, dirs, files in os.walk(resume_from):
                if os.path.basename(root) == "work":
                    for f in files:
                        dst = os.path.join(self.dir, f)
                        if not os.path.exists(dst): shutil.copy(os.path.join(root, f), dst); n += 1
            log(f"restored {n} files from {resume_from}")
        self.sp = os.path.join(self.dir, "state.json")
        self.state = json.load(open(self.sp)) if os.path.exists(self.sp) else {}

    # It takes name as input, returns a path inside the work folder.
    def p(self, name): return os.path.join(self.dir, name)
    # It takes key as input, checks whether a step is already finished and returns True or False.
    def done(self, key): return key in self.state
    # It takes key and val (value) as input, saves the result of a step and returns nothing.
    def put(self, key, val): self.state[key] = val; json.dump(self.state, open(self.sp + ".tmp", "w"), default=float); os.replace(self.sp + ".tmp", self.sp)

# It takes n_stage1 (runs in stage 1), n_stage2 (runs in stage 2), epochs (number of epochs), n_train_docs (number of training documents) and n_all_docs (number of all documents) as input, tells the progress bar how much work there is and returns nothing.
def plan(n_stage1, n_stage2, epochs, n_train_docs, n_all_docs):
    C.PROGRESS.update(total=float((n_stage1 * n_train_docs + n_stage2 * n_all_docs) * epochs), done=0.0, skipped=0.0, t0=None)

# It frees unused CPU and GPU memory and returns nothing.
def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# It takes out (output folder) as input, zips small output files for easy download and returns nothing.
def autosave(out):
    z = os.path.join(out, "all_outputs.zip"); tmp = z + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(out):
            dirs[:] = [d for d in dirs if d not in ("__pycache__", "phase2_data", ".cache")]
            for f in files:
                p = os.path.join(root, f)
                if f.endswith((".pt", ".pth", ".bin", ".tmp", ".pyc")) or f == "all_outputs.zip": continue
                zf.write(p, os.path.relpath(p, out))
    os.replace(tmp, z)

# It takes model, encs (encoded documents), cfg (settings), device and strides (window strides) as input, averages probabilities from several models and returns probabilities (or spans) per document.
def predict(model, encs, cfg, device, strides=None):
    kw = dict(bs=max(2, cfg["bs"] * 2), double=cfg.get("double", False), pad_id=cfg.get("pad_id", 0), sep_id=cfg.get("sep_id", 1), device=device)
    strides = strides or [cfg["stride"]]
    return C.predict_probs_multi(model, encs, cfg["W"], strides, **kw) if len(strides) > 1 else C.predict_probs(model, encs, cfg["W"], strides[0], **kw)

# It takes model, best (best weights) and ex (last and averaged weights) as input, groups the best, final and averaged weights and returns a dictionary with best, final and avg weights.
def _states(model, best, ex):
    return {"best": best, "final": ex["last"], "avg": ex["avg"]}

# It takes W (work folder), name, setup (function that builds the model), cfg (settings), encs (encoded documents), tr_ids (training document ids), va_ids (hold-out document ids), test_encs (encoded test documents), device, tta (prediction strides), swa_last (epochs to average), log (print function), soft_encs (documents with teacher probabilities) and extra (extra settings) as input, trains on the train part and scores on the hold-out every epoch and returns a results of the run.
def stage1(W, name, setup, cfg, encs, tr_ids, va_ids, test_encs, device, tta, swa_last=5, log=print, soft_encs=None, extra=None):
    key = f"s1:{name}"
    if W.done(key):
        log(f"[{key}] already done"); C.PROGRESS["skipped"] += cfg["epochs"] * (len(tr_ids) + len(extra or [])); return W.state[key]
    t0 = time.time()
    C.CTX_TOKENS = cfg.get("ctx", 0)
    c = dict(cfg, return_last=True, swa_last=swa_last, patience=0, ckpt_path=W.p(f"ckpt_{name}_s1.pt"), save_every=W.save_every)
    model, groups, info = setup(); model.to(device)
    tr_n, all_n = C.count_params(model); log(f"[{name}] trainable {tr_n:,} of {all_n:,} parameters")
    hist, best_ep, best, ex = C.fit(model, [encs[k] for k in tr_ids] + list(extra or []), [encs[k] for k in va_ids], c, groups, device=device, log=log)
    vencs = [encs[k] for k in va_ids]; gold = {k: encs[k].spans for k in va_ids}
    res = dict(name=name, info=info, trainable_params=tr_n, total_params=all_n, history=hist, best_epoch=best_ep, holdout={}, secs=0)
    for wn, st in _states(model, best, ex).items():
        if st is None: continue
        C.load_best(model, st)
        pv = predict(model, vencs, c, device, tta)
        C.save_probs(W.p(f"{name}_{wn}_holdout_probs.npz"), vencs, pv)
        sc = C.strict_scores(gold, C.probs_to_spans(vencs, pv))
        res["holdout"][wn] = dict(macro_F1=sc["macro_F1"], macro_P=sc["macro_P"], macro_R=sc["macro_R"], micro_F1=sc["micro_F1"],
                                  per_label={l: sc["per_label"][l]["F1"] for l in C.LABELS})
        C.print_scores(sc, f"[{name}] HOLD-OUT with {wn.upper()} weights" + (f" (epoch {best_ep})" if wn == "best" else ""))
        if test_encs: C.save_probs(W.p(f"{name}_{wn}_dev_test_probs.npz"), test_encs, predict(model, test_encs, c, device, tta))
    if soft_encs:
        C.load_best(model, ex["last"]); ps = predict(model, soft_encs, c, device, None)
        C.save_probs(W.p(f"{name}_soft.npz"), soft_encs, ps)
    res["secs"] = time.time() - t0
    W.put(key, res); del model, groups, best, ex; free()
    if os.path.exists(c["ckpt_path"]): os.remove(c["ckpt_path"])
    autosave(W.out); return res

# It takes W (work folder), name, setup (function that builds the model), cfg (settings), encs (encoded documents), all_ids (ids of all documents), test_encs (encoded test documents), seeds (random seeds), epochs (number of epochs), best_ep (best epoch), device, tta (prediction strides), swa_last (epochs to average), log (print function) and extra (extra settings) as input, trains again on all labelled documents and returns nothing.
def stage2(W, name, setup, cfg, encs, all_ids, test_encs, seeds, epochs, best_ep, device, tta, swa_last=5, log=print, extra=None):
    C.CTX_TOKENS = cfg.get("ctx", 0)
    for seed in seeds:
        key = f"s2:{name}:{seed}"
        if W.done(key):
            log(f"[{key}] already done"); C.PROGRESS["skipped"] += epochs * (len(all_ids) + len(extra or [])); continue
        t0 = time.time()
        # It takes ep (epoch), model and seed (random seed) as input, returns nothing; it is the weights from the best epoch.
        def snap(ep, model, seed=seed):
            C.save_probs(W.p(f"{name}_best_full_s{seed}.npz"), test_encs, predict(model, test_encs, cfg, device, tta))
        c = dict(cfg, seed=seed, epochs=epochs, select="last", return_last=True, swa_last=swa_last, patience=0,
                 ckpt_path=W.p(f"ckpt_{name}_s2_{seed}.pt"), save_every=W.save_every, snapshot_epochs=[best_ep] if 0 < best_ep < epochs else [], snapshot_fn=snap)
        model, groups, _ = setup(); model.to(device)
        hist, _, last, ex = C.fit(model, [encs[k] for k in all_ids] + list(extra or []), None, c, groups, device=device, log=log)
        for wn, st in (("final", ex["last"]), ("avg", ex["avg"])):
            if st is None: continue
            C.load_best(model, st); C.save_probs(W.p(f"{name}_{wn}_full_s{seed}.npz"), test_encs, predict(model, test_encs, c, device, tta))
        if not os.path.exists(W.p(f"{name}_best_full_s{seed}.npz")):
            shutil.copy(W.p(f"{name}_final_full_s{seed}.npz"), W.p(f"{name}_best_full_s{seed}.npz"))
        W.put(key, dict(secs=time.time() - t0, epochs=epochs)); log(f"[{key}] done in {(time.time()-t0)/60:.1f} min")
        del model, groups, last, ex; free()
        if os.path.exists(c["ckpt_path"]): os.remove(c["ckpt_path"])
        autosave(W.out)

# It takes path (file path) as input, loads saved probabilities from a file and returns probabilities per document.
def _load(path):
    z = np.load(path); return {str(i): z[f"p_{i}"].astype(np.float32) for i in z["ids"]}

# It takes W (work folder), name, docs (documents), va_ids (hold-out document ids), test_docs (test documents) and log (print function) as input, averages saved test predictions and writes the submission files and returns a list of written submission files.
def assemble(W, name, docs, va_ids, test_docs, log=print):
    segs = {k: C.segment(docs[k]["text"]) for k in va_ids}; gold = {k: docs[k]["spans"] for k in va_ids}
    tsegs = {k: C.segment(v["text"]) for k, v in test_docs.items()} if test_docs else {}
    index, files = [], []
    for wn in WEIGHTS:
        hp = W.p(f"{name}_{wn}_holdout_probs.npz")
        if not os.path.exists(hp): continue
        shutil.copy(hp, os.path.join(W.out, f"{name}_{wn}_holdout_probs.npz"))
        hv = D.score_variants(_load(hp), gold, segs)
        fulls = sorted(glob.glob(W.p(f"{name}_{wn}_full_s*.npz")))
        src, kind = fulls, "full"
        if not fulls: src, kind = [W.p(f"{name}_{wn}_dev_test_probs.npz")], "dev"
        if not os.path.exists(src[0]): continue
        tp = [_load(f) for f in src]; probs = {i: sum(t[i] for t in tp) / len(tp) for i in tp[0]}
        np.savez_compressed(os.path.join(W.out, f"{name}_{wn}_{kind}_test_probs.npz"), ids=np.array(list(probs)), **{f"p_{i}": p.astype(np.float16) for i, p in probs.items()})
        for vn, kw in D.VARIANTS.items():
            pred = D.decode_all(probs, tsegs, **kw)
            fn = os.path.join(W.out, "submissions", f"{name}__{kind}_{wn}__{vn}_submission.zip")
            C.write_submission(pred, test_docs, fn); files.append(fn)
            index.append(dict(file=os.path.basename(fn), weights=wn, variant=vn, trained_on=("all 70 docs, %d seeds" % len(src)) if kind == "full" else "train part only",
                              holdout_macro_F1_of_dev_model=round(hv[vn], 4), n_test_entities=sum(len(v) for v in pred.values())))
    json.dump(index, open(os.path.join(W.out, f"{name}_submissions_index.json"), "w"), indent=1)
    log(f"\n[{name}] submissions written (hold-out score = the dev model with the same weights and decoding, 14 documents -> +-0.03 noise):")
    for r in sorted(index, key=lambda r: -r["holdout_macro_F1_of_dev_model"])[:8]: log("   %.4f  %s" % (r["holdout_macro_F1_of_dev_model"], r["file"]))
    return index

# It takes test_docs (test documents), test_encs (encoded test documents), probs (probabilities), thr (threshold), tok (tokenizer) and log (print function) as input, makes pseudo-labelled test documents from model predictions and returns a list of pseudo-labelled documents.
def make_pseudo(test_docs, test_encs, probs, thr, tok=None, log=print):
    import copy
    out = []; kept = tot = ent = 0
    for e in test_encs:
        ne = copy.copy(e); ne.spans = D.decode_doc(probs[e.id], e.ws, e.we)
        tg = C.tag_words(e.ws, e.we, ne.spans); conf = probs[e.id].max(1)
        ne.tags = np.where(conf >= thr, tg, -100).astype(np.int64); ne.soft = None; ne.id = "pl_" + e.id
        kept += int((ne.tags >= 0).sum()); tot += len(ne.tags); ent += int((ne.tags > 0).sum()); out.append(ne)
    log(f"pseudo-labels: {kept}/{tot} words kept (confidence >= {thr}), {ent} of them inside entities")
    return out

# It takes W (work folder), name, setup (function that builds the model), cfg (settings), encs (encoded documents), tr_ids (training document ids), va_ids (hold-out document ids), docs (documents), test_docs (test documents), test_encs (encoded test documents), base (name of the base run), device, tta (prediction strides), swa_last (epochs to average), thr (threshold), seeds (random seeds), epochs (number of epochs) and log (print function) as input, trains again using pseudo-labelled test documents and returns a list of written submission files.
def pseudo_stage(W, name, setup, cfg, encs, tr_ids, va_ids, docs, test_docs, test_encs, base, device, tta, swa_last, thr, seeds, epochs, log=print):
    dev_p = _load(W.p(f"{name}_avg_dev_test_probs.npz"))
    r = stage1(W, name + "_pl", setup, cfg, encs, tr_ids, va_ids, test_encs, device, tta, swa_last, extra=make_pseudo(test_docs, test_encs, dev_p, thr))
    b0 = base["holdout"].get("avg", base["holdout"]["final"])["macro_F1"]; b1 = r["holdout"]["avg"]["macro_F1"]
    log(f"== PSEUDO-LABEL CHECK {name} (hold-out, avg weights): without {b0:.4f} -> with {b1:.4f} ({b1-b0:+.4f}; 14 docs = +-0.03 noise)")
    fulls = sorted(glob.glob(W.p(f"{name}_avg_full_s*.npz")))
    fp = {i: sum(p[i] for p in ps) / len(ps) for ps in [[_load(f) for f in fulls]] for i in ps[0]}
    stage2(W, name + "_pl", setup, cfg, encs, sorted(docs), test_encs, seeds, epochs, r["best_epoch"], device, tta, swa_last, extra=make_pseudo(test_docs, test_encs, fp, thr))
    log(f"pseudo-label submissions: {name}_pl__full_*  - submit them only if the check above is positive.")
    return assemble(W, name + "_pl", docs, va_ids, test_docs)

Writing ner_opt.py


In [8]:
# Import the Legal Transformer model code
import arch_legal_transformer as A

In [9]:
# Load the NER data and make train / hold-out split
import sys, os, json, time, glob, random
import numpy as np, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
import ner_core as C, ner_peft as P, ner_opt as O
C.seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPUs:", torch.cuda.device_count())
os.makedirs(OUT, exist_ok=True)

if AUTO_DOWNLOAD: C.ensure_phase2_data()
docs = C.load_train_docs()
test_docs = C.load_test_docs()
if SMOKE:
    keep = sorted(docs)[:6]
    docs = {k: dict(docs[k], text=docs[k]["text"][:SMOKE_CHARS], spans=[s for s in docs[k]["spans"] if s[1] <= SMOKE_CHARS]) for k in keep}
    if test_docs: test_docs = {k: dict(v, text=v["text"][:SMOKE_CHARS]) for k, v in list(test_docs.items())[:2]}
    EPOCHS, N_VAL_DOCS, EVAL_FROM, SWA_LAST, FULL_SEEDS, TTA_STRIDES = 2, 2, 1, 1, FULL_SEEDS[:1], TTA_STRIDES[:2]
print(f"{len(docs)} labelled docs, test docs: {len(test_docs) if test_docs else 'NOT FOUND (attach Test/fold3-text.json)'}")

from collections import Counter
cnt = Counter(l for d in docs.values() for _, _, l in d["spans"])
nd = {l: sum(1 for d in docs.values() if any(s[2] == l for s in d["spans"])) for l in C.LABELS}
print("%-10s %7s %9s" % ("label", "entities", "in docs"))
for l in C.LABELS: print("%-10s %7d %6d/%d" % (l, cnt[l], nd[l], len(docs)))
orc = C.oracle_check(docs, list(docs))
print(f"\nORACLE: if the word tagger were perfect, strict macro-F1 would be {orc['macro_F1']:.4f}  (upper bound set by the word segmentation)")
print("labels the segmentation cannot reproduce exactly:", {l: round(orc['per_label'][l]['F1'], 3) for l in C.LABELS if orc['per_label'][l]['F1'] < 0.999})

tr_ids, va_ids = C.make_split(docs, n_val=N_VAL_DOCS, seed=SEED)
vc = Counter(l for k in va_ids for _, _, l in docs[k]["spans"])
print(f"\ndocument-level split: {len(tr_ids)} train docs / {len(va_ids)} hold-out docs; hold-out entities per label:", dict(vc))

device: cuda | GPUs: 2


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

  downloaded fold1-text.json -> /kaggle/working/phase2_data/Phase_2_LNER/Train/fold1-text.json


fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

  downloaded fold1-gold.json -> /kaggle/working/phase2_data/Phase_2_LNER/Train/fold1-gold.json


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

... (3 outputs left out)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

  downloaded labels.txt -> /kaggle/working/phase2_data/Phase_2_LNER/Train/labels.txt


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

  downloaded fold3-text.json -> /kaggle/working/phase2_data/Phase_2_LNER/Test/fold3-text.json
70 labelled docs, test docs: 35
label      entities   in docs
APP            407     63/70
RESP           381     63/70
A.COUNSEL      195     49/70
R.COUNSEL      184     47/70
JUDGE          245     66/70
WIT            187     12/70
AUTH          1003     43/70
COURT         1013     66/70
STAT          1260     65/70
PREC          1183     66/70
DATE          1575     70/70
CASENO         760     65/70

ORACLE: if the word tagger were perfect, strict macro-F1 would be 0.9944  (upper bound set by the word segmentation)
labels the segmentation cannot reproduce exactly: {'APP': 0.985, 'RESP': 0.995, 'AUTH': 0.991, 'COURT': 0.982, 'STAT': 0.984, 'PREC': 0.995}

document-level split: 56 train docs / 14 hold-out docs; hold-out entities per label: {'COURT': 219, 'CASENO': 173, 'APP': 121, 'RESP': 78, 'STAT': 256, 'DATE': 329, 'AUTH': 209, 'JUDGE': 50, 'PREC': 295, 'A.COUNSEL': 45, 'R.COUNSEL': 48

In [10]:
# Build training config and learning-rate groups
# It takes bb (pre-trained backbone) and tok (tokenizer) as input, builds the training settings and returns a dictionary of training settings.
def make_cfg(bb, tok):
    W = min(WIN_TOKENS, (bb.max_len - 1) // 2 if DOUBLE_PASS else bb.max_len)
    return dict(epochs=EPOCHS, W=W, stride=min(STRIDE, max(1, W // 2)) if W < WIN_TOKENS else STRIDE, bs=MICRO_BS, accum=ACCUM, double=DOUBLE_PASS,
                rare_dup=RARE_DUP, class_power=CLASS_POWER, patience=PATIENCE, wd=WEIGHT_DECAY, seed=SEED, eval_from=EVAL_FROM, bs_state=dict(bs=MICRO_BS),
                pad_id=tok.pad_id if tok.pad_id is not None else 0, sep_id=tok.eos_id if tok.eos_id is not None else 1, amp=True, label_smooth=LABEL_SMOOTH)

# It takes model, bb_lr (backbone learning rate), head_lr (head learning rate) and wd (weight decay) as input, sets learning rates for the new head and the backbone and returns a list of parameter groups with learning rates.
def head_groups(model, bb_lr, head_lr=None, wd=WEIGHT_DECAY):
    head = [p for n, p in model.named_parameters() if not n.startswith("bb_model.") and p.requires_grad]
    bbp = [p for n, p in model.named_parameters() if n.startswith("bb_model.") and p.requires_grad]
    g = [dict(params=head, lr=head_lr or LR_HEAD, weight_decay=wd)]
    if bbp: g.append(dict(params=bbp, lr=bb_lr, weight_decay=wd))
    return g

In [11]:
# Load the pre-trained model and tokenize the documents
bb0 = A.load_backbone(CKPT, device="cpu")
tok = A.load_tokenizer(TOK)
print(f"backbone: {bb0.n_params/1e6:.1f}M parameters, d_model {bb0.d_model}, max_len {bb0.max_len} | tokenizer vocab {tok.vocab_size}")
CFG = make_cfg(bb0, tok); print("training config:", CFG)
encs = {e.id: e for e in C.build_encs(docs, sorted(docs), tok)}
print("tokenizer:", C.token_stats(list(encs.values())))
test_encs = C.build_encs(test_docs, sorted(test_docs), tok) if test_docs else None
del bb0
# It loads a fresh copy of the pre-trained model and returns the backbone.
def new_backbone():
    bb = A.load_backbone(CKPT, device="cpu")
    if BIDIRECTIONAL: bb.set_bidirectional(True)
    return bb

backbone: 271.2M parameters, d_model 896, max_len 640 | tokenizer vocab 12000
training config: {'epochs': 20, 'W': 512, 'stride': 256, 'bs': 8, 'accum': 2, 'double': False, 'rare_dup': 2, 'class_power': 0.5, 'patience': 0, 'wd': 0.01, 'seed': 42, 'eval_from': 2, 'bs_state': {'bs': 8}, 'pad_id': 0, 'sep_id': 1, 'amp': True, 'label_smooth': 0.05}
encoded 70 docs, 505,231 tokens, 449,428 words in 2s
tokenizer: {'tokens_per_char': 0.22789064480211588, 'chars_per_token': 4.388069553571755}
encoded 35 docs, 269,294 tokens, 239,512 words in 1s


In [12]:
# Set up the model with LoRMA
# It takes v (variant settings) as input, returns a function that builds the model for one setting.
def make_setup(v):
    # It uses values from the outer function, builds the model, its learning rates and info and returns (model, parameter groups, info).
    def setup():
        bb = new_backbone()
        n = P.apply_linear_wrappers(bb, "lorma", bb.linear_names(), r=v["rank"], alpha=v["rank"])
        model = C.NERModel(bb, GRU_HIDDEN, dropout=HEAD_DROPOUT, double=DOUBLE_PASS, sep_id=CFG["sep_id"])
        return model, head_groups(model, v["lr"]), dict(technique="LoRMA", wrapped_linears=n, **v)
    return setup

## Run: hold-out comparison of all variants -> best two on all data -> ensemble

In [13]:
# Train all variants, pick the best, and build the ensemble
O_work = O.Work(OUT, RESUME_FROM, SAVE_EVERY_MIN * 60)
O.plan(len(VARIANTS[:1] if SMOKE else VARIANTS), N_FINAL_VARIANTS * len(FULL_SEEDS), EPOCHS, len(tr_ids), len(docs))
# It takes v (variant settings) as input, returns a dictionary of settings; it is the training settings for one variant.
def cfg_for(v):
    c = dict(CFG, ctx=v["ctx"])
    if v["ctx"]: c["W"] = CFG["W"] - v["ctx"] - 1; c["stride"] = max(64, int(CFG["stride"] * c["W"] / CFG["W"]) if False else c["W"] // 2)
    return c
# It takes v (variant settings) as input, returns a list of strides; it is the window strides used at prediction time for one variant.
def tta_for(v):
    W = CFG["W"] - (v["ctx"] + 1 if v["ctx"] else 0)
    return sorted({max(64, min(W, int(s * W / CFG["W"]))) for s in TTA_STRIDES}, reverse=True)
SEL = {}
for v in (VARIANTS[:1] if SMOKE else VARIANTS):
    SEL[v["tag"]] = O.stage1(O_work, f"{NAME}_{v['tag']}", make_setup(v), cfg_for(v), encs, tr_ids, va_ids, test_encs, DEVICE, tta_for(v), SWA_LAST)
    h = SEL[v["tag"]]["holdout"]
    print(f"== {v['tag']}: hold-out macro-F1  best {h['best']['macro_F1']:.4f} | final {h['final']['macro_F1']:.4f} | avg {h.get('avg', h['final'])['macro_F1']:.4f}")
    O.assemble(O_work, f"{NAME}_{v['tag']}", docs, va_ids, test_docs)
score = lambda t: SEL[t]["holdout"].get("avg", SEL[t]["holdout"]["final"])["macro_F1"]
top = sorted(SEL, key=score, reverse=True)[:(1 if SMOKE else N_FINAL_VARIANTS)]
print("variants retrained on all 70 documents:", top)
names = []
for t in top:
    v = next(x for x in VARIANTS if x["tag"] == t); nm = f"{NAME}_{t}"; names.append(nm)
    O.stage2(O_work, nm, make_setup(v), cfg_for(v), encs, sorted(docs), test_encs, FULL_SEEDS, EPOCHS, SEL[t]["best_epoch"], DEVICE, tta_for(v), SWA_LAST)
    O.assemble(O_work, nm, docs, va_ids, test_docs)
import ner_decode as D
segs_h = {k: C.segment(docs[k]["text"]) for k in va_ids}; gold_h = {k: docs[k]["spans"] for k in va_ids}
tsegs = {k: C.segment(x["text"]) for k, x in test_docs.items()} if test_docs else {}
# It takes ps (list of probabilities) as input, averages several probability dictionaries and returns averaged probabilities.
def avg_probs(ps): return {i: sum(p[i] for p in ps) / len(ps) for i in ps[0]}
ENS = []
for wn in O.WEIGHTS:
    hp = [O._load(O_work.p(f"{n}_{wn}_holdout_probs.npz")) for n in names if os.path.exists(O_work.p(f"{n}_{wn}_holdout_probs.npz"))]
    tf = [sorted(glob.glob(O_work.p(f"{n}_{wn}_full_s*.npz"))) for n in names]
    tp = [avg_probs([O._load(f) for f in fs]) for fs in tf if fs]
    if not hp or not tp or not test_docs: continue
    hv = D.score_variants(avg_probs(hp), gold_h, segs_h); probs = avg_probs(tp)
    if wn == "avg": FULL_AVG = probs
    for vn, kw in D.VARIANTS.items():
        fn = f"{OUT}/submissions/{NAME}_ENSEMBLE__full_{wn}__{vn}_submission.zip"
        C.write_submission(D.decode_all(probs, tsegs, **kw), test_docs, fn)
        ENS.append((hv[vn], os.path.basename(fn)))
for sc_, fn in sorted(ENS, reverse=True)[:10]: print("%.4f  %s" % (sc_, fn))
json.dump(sorted(ENS, reverse=True), open(f"{OUT}/{NAME}_ensemble_index.json", "w"), indent=1)
if PSEUDO and test_docs and "FULL_AVG" in globals():
    t0 = top[0]; v0 = next(x for x in VARIANTS if x["tag"] == t0); nm0 = f"{NAME}_{t0}_pl"
    dev_avg = avg_probs([O._load(O_work.p(f"{n}_avg_dev_test_probs.npz")) for n in names])
    pl_dev = O.make_pseudo(test_docs, test_encs, dev_avg, PSEUDO_THR, tok)
    r = O.stage1(O_work, nm0, make_setup(v0), cfg_for(v0), encs, tr_ids, va_ids, test_encs, DEVICE, tta_for(v0), SWA_LAST, extra=pl_dev)
    b0 = SEL[t0]["holdout"]["avg"]["macro_F1"]; b1 = r["holdout"]["avg"]["macro_F1"]
    print(f"== PSEUDO-LABEL CHECK (hold-out, avg weights): without {b0:.4f}  ->  with {b1:.4f}  ({b1-b0:+.4f}; 14 docs = +-0.03 noise)")
    pl_full = O.make_pseudo(test_docs, test_encs, FULL_AVG, PSEUDO_THR, tok)
    O.stage2(O_work, nm0, make_setup(v0), cfg_for(v0), encs, sorted(docs), test_encs, FULL_SEEDS[:PSEUDO_SEEDS], EPOCHS, r["best_epoch"], DEVICE, tta_for(v0), SWA_LAST, extra=pl_full)
    O.assemble(O_work, nm0, docs, va_ids, test_docs)
    print("pseudo-label submissions are named ..._pl__full_<weights>__<variant>; submit them only if the check above is positive.")

[lorma_lt_best_r32_ctx] trainable 15,719,961 of 286,900,633 parameters
windows/epoch 3278, optimizer steps/epoch 204, total 4080, micro-batch 8 x accum 2


epoch 1/20:   0%|          | 0/204 [00:00<?, ?it/s]

epoch 1  train_ce 0.7198  secs 265.5017  | this run: epoch 1/20, ETA 1h24m  [overall 1% | elapsed 0h04m | ETA 6h33m]


epoch 2/20:   0%|          | 0/204 [00:00<?, ?it/s]

epoch 2  train_ce 0.4340  secs 269.3240  val_macro_F1 0.5280  val_micro_F1 0.6121  | this run: epoch 2/20, ETA 1h27m  [overall 2% | elapsed 0h09m | ETA 6h49m]


epoch 3/20:   0%|          | 0/204 [00:00<?, ?it/s]

... (137 outputs left out)

epoch 19/20:   0%|          | 0/209 [00:00<?, ?it/s]

epoch 19  train_ce 0.3805  secs 275.4764  | this run: epoch 19/20, ETA 0h04m  [overall 99% | elapsed 6h41m | ETA 0h05m]


epoch 20/20:   0%|          | 0/209 [00:00<?, ?it/s]

epoch 20  train_ce 0.3804  secs 275.6815  | this run: epoch 20/20, ETA 0h00m  [overall 100% | elapsed 6h46m | ETA 0h00m]
[s2:lorma_lt_best_r32_ctx0:1] done in 101.9 min
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_best__viterbi_submission.zip: 35 docs, 4003 entities
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_best__ob-0.5_submission.zip: 35 docs, 4242 entities
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_best__ob-1.0_submission.zip: 35 docs, 4467 entities
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_best__ob-1.5_submission.zip: 35 docs, 4655 entities
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_best__topk2_submission.zip: 35 docs, 3997 entities
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_best__topp0.9_submission.zip: 35 docs, 4003 entities
wrote /kaggle/working/submissions/lorma_lt_best_r32_ctx0__full_final__viterbi_submission.zip: 35 docs, 4032 entities
wrote /kaggle/working/s

## Summary

In [14]:
# Show results table and plots
st = json.load(open(f"{OUT}/work/state.json"))
print("%-34s %9s %9s %9s %9s" % ("hold-out strict macro-F1", "best", "final", "avg", "best ep"))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for k, r in st.items():
    if not k.startswith("s1:"): continue
    h = r["holdout"]; print("%-34s %9.4f %9.4f %9.4f %9d" % (k[3:], h["best"]["macro_F1"], h["final"]["macro_F1"], h.get("avg", h["final"])["macro_F1"], r["best_epoch"]))
    ax[0].plot([x["epoch"] for x in r["history"] if "val_macro_F1" in x], [x["val_macro_F1"] for x in r["history"] if "val_macro_F1" in x], label=k[3:])
ax[0].set_title("hold-out strict macro-F1 per epoch"); ax[0].set_xlabel("epoch"); ax[0].legend()
for i, (k, r) in enumerate([(k, r) for k, r in st.items() if k.startswith("s1:")]):
    ax[1].bar(np.arange(12) + i * 0.8 / max(1, sum(1 for x in st if x.startswith("s1:"))), [r["holdout"].get("avg", r["holdout"]["final"])["per_label"][l] for l in C.LABELS], 0.8 / max(1, sum(1 for x in st if x.startswith("s1:"))), label=k[3:])
ax[1].set_xticks(np.arange(12)); ax[1].set_xticklabels(C.LABELS, rotation=45); ax[1].set_title("hold-out F1 per label (avg weights)"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/summary.png", dpi=110); plt.show()
O.autosave(OUT)
print("\nONE-CLICK DOWNLOAD: /kaggle/working/all_outputs.zip  (submissions, probabilities, reports).")
print("Submit the files in /kaggle/working/submissions/ - names say what they are: <model>__full_<weights>__<decoding>_submission.zip (full = trained on all 70 docs).")
print("Keep the *_holdout_probs.npz and *_full_test_probs.npz files for ensembling.")

hold-out strict macro-F1                best     final       avg   best ep
lorma_lt_best_r32_ctx                 0.5523    0.5468    0.5462        11
lorma_lt_best_r32_ctx0                0.5691    0.5418    0.5470         7

ONE-CLICK DOWNLOAD: /kaggle/working/all_outputs.zip  (submissions, probabilities, reports).
Submit the files in /kaggle/working/submissions/ - names say what they are: <model>__full_<weights>__<decoding>_submission.zip (full = trained on all 70 docs).
Keep the *_holdout_probs.npz and *_full_test_probs.npz files for ensembling.


## Zip everything

In [15]:
# Zip all outputs
import os, zipfile, time
_out = globals().get("OUT", "/kaggle/working")
_tmp = os.path.join(_out, "_zipping.tmp"); _dst = os.path.join(_out, "everything_%s.zip" % time.strftime("%Y%m%d_%H%M"))
n = 0
with zipfile.ZipFile(_tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as z:
    for root, dirs, files in os.walk(_out):
        dirs[:] = [d for d in dirs if d not in ("__pycache__", ".ipynb_checkpoints")]
        for f in files:
            p = os.path.join(root, f)
            if f in ("_zipping.tmp",) or (f.startswith("everything_") and f.endswith(".zip")): continue
            try: z.write(p, os.path.relpath(p, _out)); n += 1
            except Exception as e: print("skipped", p, e)
os.replace(_tmp, _dst)
print("zipped %d files -> %s (%.1f MB). Download it from the Output tab." % (n, _dst, os.path.getsize(_dst) / 1e6))

zipped 137 files -> /kaggle/working/everything_20261010_2246.zip (504.1 MB). Download it from the Output tab.
